In [ ]:
import os

folder = r"C:\Users\acer\OneDrive\Desktop\ZIDIO\churn_customer_prediction"

print("Folder exists:", os.path.exists(folder))
if os.path.exists(folder):
    for f in os.listdir(folder):
        print(f)

In [ ]:
import pandas as pd

folder = r"C:\Users\acer\OneDrive\Desktop\ZIDIO\churn_customer_prediction"
df_churn1 = pd.read_csv(folder + r"\Customer Churn.csv")
df_churn2 = pd.read_csv(folder + r"\Customer-Churn-Records.csv")
df_churn3 = pd.read_excel(folder + r"\churn.xlsx")
df_telco  = pd.read_excel(folder + r"\Telco_customer_churn.xlsx")

print("Customer Churn.csv loaded")
print("Customer-Churn-Records.csv loaded")
print("churn.xlsx loaded")
print("Telco_customer_churn.xlsx loaded")

In [ ]:
datasets = {
    "Customer churn.csv": df_churn1,
    "Customer-Churn-Records.csv": df_churn2,
    "churn.xlsx": df_churn3,
    "Telco_customer_churn.xlsx": df_telco,
}

for name, df in datasets.items():
    print("=" * 70)
    print(name)
    print("Shape (rows, columns):", df.shape)
    print("Columns:", list(df.columns))

In [ ]:
df = df_churn1.copy()
print("Data types")
print(df.dtypes)

In [ ]:
df.isnull().sum()

In [ ]:
(df["TotalCharges"].astype(str).str.strip() == "").sum()

In [ ]:
(df["Churn"].value_counts())

In [ ]:

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing TotalCharges now:", df["TotalCharges"].isnull().sum())
print("New data type:", df["TotalCharges"].dtype)

print("\n---- The rows with missing TotalCharges ----")
display(df[df["TotalCharges"].isnull()][["customerID", "tenure", "MonthlyCharges", "Contract", "Churn"]])

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [ ]:
print("Missing TotalCharges:", df["TotalCharges"].isnull().sum())
print("Total missing in whole data:", df.isnull().sum().sum())
print("Duplicate customerID:", df["customerID"].duplicated().sum())
print("Duplicate full rows:", df.duplicated().sum())
print("Shape:", df.shape)


In [ ]:
import pandas as pd

folder = r"C:\Users\acer\OneDrive\Desktop\ZIDIO\churn_customer_prediction"

df = pd.read_csv(folder + r"\Customer Churn.csv")

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("Missing TotalCharges:", df["TotalCharges"].isnull().sum())
print("Total missing in whole data:", df.isnull().sum().sum())
print("Duplicate customerID:", df["customerID"].duplicated().sum())
print("Shape:", df.shape)

save_path = folder + r"\Customer_Churn_Python_Clean.csv"
df.to_csv(save_path, index=False)
print("Saved:", save_path)

check = pd.read_csv(save_path)
print("Reloaded shape:", check.shape)
print("TotalCharges type:", check["TotalCharges"].dtype)
print("Reloaded missing values:", check.isnull().sum().sum())

In [ ]:
import pandas as pd

folder = r"C:\Users\acer\OneDrive\Desktop\ZIDIO\churn_customer_prediction"
df = pd.read_csv(folder + r"\Customer_Churn_Python_Clean.csv")
print(df.shape)

In [ ]:
text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    print(f"{col}  ->  {df[col].nunique()} unique values")
    if col != "customerID":
        print("   ", list(df[col].unique()))

In [ ]:
df_enc = df.copy() 

yes_no_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]
for col in yes_no_cols:
    df_enc[col] = df_enc[col].map({"No": 0, "Yes": 1})

df_enc["gender"] = df_enc["gender"].map({"Female": 0, "Male": 1})

cols = ["gender"] + yes_no_cols
print(df_enc[cols].dtypes)
print("\nChurn counts:")
print(df_enc["Churn"].value_counts())
print("\nMissing values in these columns:", df_enc[cols].isnull().sum().sum())
display(df_enc[cols].head())

In [ ]:
df_enc = df_enc.drop(columns=["customerID"])

cat_cols = ["MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
            "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
            "Contract", "PaymentMethod"]

df_enc = pd.get_dummies(df_enc, columns=cat_cols, drop_first=True, dtype=int)

print("Shape:", df_enc.shape)
print("Text columns left:", len(df_enc.select_dtypes(include="object").columns))
print("Missing values:", df_enc.isnull().sum().sum())
print("\nColumns:")
print(list(df_enc.columns))

In [ ]:
import numpy as np

df_feat = df_enc.copy()

service_cols = ["OnlineSecurity_Yes", "OnlineBackup_Yes", "DeviceProtection_Yes",
                "TechSupport_Yes", "StreamingTV_Yes", "StreamingMovies_Yes"]

df_feat["services_count"] = df_feat[service_cols].sum(axis=1)
df_feat["is_new_customer"] = (df_feat["tenure"] <= 12).astype(int)
df_feat["avg_monthly_spend"] = np.where(df_feat["tenure"] > 0,
                                        df_feat["TotalCharges"] / df_feat["tenure"],
                                        df_feat["MonthlyCharges"])

print("Shape:", df_feat.shape)
print("Missing values:", df_feat.isnull().sum().sum())
display(df_feat[["services_count", "is_new_customer", "avg_monthly_spend"]].describe())

In [ ]:
from sklearn.model_selection import train_test_split


df_feat.to_csv(folder + r"\Customer_Churn_Features.csv", index=False)

X = df_feat.drop(columns=["Churn"])
y = df_feat["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X shape:", X.shape, "| y shape:", y.shape)
print("Train rows:", X_train.shape[0], "| Test rows:", X_test.shape[0])
print("Churn rate in train:", round(y_train.mean() * 100, 2), "%")
print("Churn rate in test :", round(y_test.mean() * 100, 2), "%")

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = ["tenure", "MonthlyCharges", "TotalCharges",
            "services_count", "avg_monthly_spend"]

scaler = StandardScaler()

X_train_s = X_train.copy()
X_test_s = X_test.copy()

X_train_s[num_cols] = scaler.fit_transform(X_train[num_cols])   
X_test_s[num_cols] = scaler.transform(X_test[num_cols])         

print("Train shape:", X_train_s.shape, "| Test shape:", X_test_s.shape)
print("\nTrain: mean and std after scaling")
display(X_train_s[num_cols].describe().loc[["mean", "std"]].round(3))
print("Test: mean and std after scaling")
display(X_test_s[num_cols].describe().loc[["mean", "std"]].round(3))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)


folder = r"C:\Users\acer\OneDrive\Desktop\ZIDIO\churn_customer_prediction"
df_feat = pd.read_csv(folder + r"\Customer_Churn_Features.csv")


X = df_feat.drop(columns=["Churn"])
y = df_feat["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


num_cols = ["tenure", "MonthlyCharges", "TotalCharges",
            "services_count", "avg_monthly_spend"]
scaler = StandardScaler()
X_train_s = X_train.copy()
X_test_s = X_test.copy()
X_train_s[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_s[num_cols] = scaler.transform(X_test[num_cols])

print(df_feat.shape, X_train_s.shape, X_test_s.shape)


log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_s, y_train)

pred = log_model.predict(X_test_s)
prob = log_model.predict_proba(X_test_s)[:, 1]

print("\nAccuracy:", round(accuracy_score(y_test, pred), 4))
print("ROC-AUC :", round(roc_auc_score(y_test, prob), 4))
print("\nConfusion matrix (rows = real, columns = predicted):")
print(confusion_matrix(y_test, pred))
print("\nDetailed report:")
print(classification_report(y_test, pred, target_names=["Stayed (0)", "Churned (1)"]))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"roc_auc": "roc_auc", "recall": "recall", "precision": "precision"}

models = {
    "Logistic Regression": (LogisticRegression(max_iter=1000, random_state=42), X_train_s),
    "Random Forest": (RandomForestClassifier(n_estimators=200, random_state=42), X_train),
}

rows = []
for name, (model, X_used) in models.items():
    res = cross_validate(model, X_used, y_train, cv=cv, scoring=scoring)
    rows.append({
        "Model": name,
        "ROC-AUC": round(res["test_roc_auc"].mean(), 4),
        "Recall (churn)": round(res["test_recall"].mean(), 4),
        "Precision (churn)": round(res["test_precision"].mean(), 4),
    })

display(pd.DataFrame(rows))

In [ ]:
models2 = {
    "LogReg (default)": (LogisticRegression(max_iter=1000, random_state=42), X_train_s),
    "LogReg (balanced)": (LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42), X_train_s),
    "Random Forest (balanced)": (RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample", random_state=42), X_train),
}

scoring2 = {"roc_auc": "roc_auc", "recall": "recall", "precision": "precision", "f1": "f1"}

rows2 = []
for name, (model, X_used) in models2.items():
    res = cross_validate(model, X_used, y_train, cv=cv, scoring=scoring2)
    rows2.append({
        "Model": name,
        "ROC-AUC": round(res["test_roc_auc"].mean(), 4),
        "Recall (churn)": round(res["test_recall"].mean(), 4),
        "Precision (churn)": round(res["test_precision"].mean(), 4),
        "F1 (churn)": round(res["test_f1"].mean(), 4),
    })

display(pd.DataFrame(rows2))

In [ ]:
from xgboost import XGBClassifier

spw = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(spw, 3))

models3 = {
    "LogReg (balanced)": (LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42), X_train_s),
    "XGBoost (weighted)": (XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                                         subsample=0.8, colsample_bytree=0.8,
                                         scale_pos_weight=spw, eval_metric="logloss",
                                         random_state=42), X_train),
}

rows3 = []
for name, (model, X_used) in models3.items():
    res = cross_validate(model, X_used, y_train, cv=cv, scoring=scoring2)
    rows3.append({
        "Model": name,
        "ROC-AUC": round(res["test_roc_auc"].mean(), 4),
        "Recall (churn)": round(res["test_recall"].mean(), 4),
        "Precision (churn)": round(res["test_precision"].mean(), 4),
        "F1 (churn)": round(res["test_f1"].mean(), 4),
    })

display(pd.DataFrame(rows3))

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, recall_score, precision_score, f1_score, accuracy_score

lr_final = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr_final.fit(X_train_s, y_train)

xgb_final = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8,
                          scale_pos_weight=spw, eval_metric="logloss", random_state=42)
xgb_final.fit(X_train, y_train)

finalists = {
    "LogReg (balanced)": (lr_final, X_test_s),
    "XGBoost (weighted)": (xgb_final, X_test),
}

rows4 = []
for name, (model, X_t) in finalists.items():
    pred = model.predict(X_t)
    prob = model.predict_proba(X_t)[:, 1]
    rows4.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, prob), 4),
        "Recall (churn)": round(recall_score(y_test, pred), 4),
        "Precision (churn)": round(precision_score(y_test, pred), 4),
        "F1 (churn)": round(f1_score(y_test, pred), 4),
    })
    print(name, "- confusion matrix (rows = real, columns = predicted):")
    print(confusion_matrix(y_test, pred), "\n")

display(pd.DataFrame(rows4))

In [ ]:
import pandas as pd

coef = pd.Series(lr_final.coef_[0], index=X_train_s.columns).sort_values()

print("LOWERS churn risk (most negative):")
display(coef.head(8).round(3))

print("RAISES churn risk (most positive):")
display(coef.tail(8).sort_values(ascending=False).round(3))

In [ ]:
import numpy as np
import joblib

clean = pd.read_csv(folder + r"\Customer_Churn_Python_Clean.csv")

# score every customer with the final Logistic Regression
X_all_s = X.copy()
X_all_s[num_cols] = scaler.transform(X[num_cols])
prob_all = lr_final.predict_proba(X_all_s)[:, 1]

scored = clean[["customerID", "Contract", "InternetService", "PaymentMethod",
                "tenure", "MonthlyCharges", "TotalCharges", "Churn"]].copy()
scored["churn_probability"] = prob_all.round(4)
scored["risk_level"] = pd.cut(prob_all, bins=[0, 0.30, 0.60, 1.0],
                              labels=["Low", "Medium", "High"], include_lowest=True)
scored["data_split"] = np.where(scored.index.isin(X_test.index), "test", "train")

scored.to_csv(folder + r"\Customer_Churn_Scored.csv", index=False)
joblib.dump(lr_final, folder + r"\logreg_final_model.joblib")
joblib.dump(scaler, folder + r"\scaler_final.joblib")

print("Shape:", scored.shape)
print("\nRisk level counts:")
print(scored["risk_level"].value_counts())
print("\nReal churn rate inside each risk level (%):")
print((scored.groupby("risk_level", observed=True)["Churn"].apply(lambda s: (s == "Yes").mean() * 100)).round(1))
print("\nData split:")
print(scored["data_split"].value_counts())

In [ ]:
t = scored[scored["data_split"] == "test"].copy()
t["left"] = (t["Churn"] == "Yes").astype(int)

summary = t.groupby("risk_level", observed=True).agg(
    customers=("left", "size"),
    churners=("left", "sum"),
    churn_rate_pct=("left", lambda s: round(s.mean() * 100, 1)),
)
summary["share_of_all_churners_pct"] = (summary["churners"] / summary["churners"].sum() * 100).round(1)

print("Test customers only (never used for training):")
display(summary)
print("Total test customers:", summary["customers"].sum(), "| Total churners:", summary["churners"].sum())

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

seg_cols = ["tenure", "MonthlyCharges", "services_count"]
X_seg = X_all_s[seg_cols]

print("k | inertia | silhouette")
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_seg)
    print(k, "|", round(km.inertia_, 1), "|", round(silhouette_score(X_seg, km.labels_), 3))

In [ ]:
for k in [3, 4]:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_seg)

    prof = df_feat[["tenure", "MonthlyCharges", "services_count", "Churn"]].copy()
    prof["segment"] = km.labels_

    summary = prof.groupby("segment").agg(
        customers=("Churn", "size"),
        avg_tenure_months=("tenure", "mean"),
        avg_monthly_charge=("MonthlyCharges", "mean"),
        avg_services=("services_count", "mean"),
        churn_rate_pct=("Churn", lambda s: s.mean() * 100),
    ).round(1)

    print(f"--- k = {k} (customers add up to {summary['customers'].sum()}) ---")
    display(summary)
    

In [ ]:
km4 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_seg)

tmp = df_feat[["tenure", "MonthlyCharges"]].copy()
tmp["seg"] = km4.labels_
means = tmp.groupby("seg").mean()

names = {s: ("Loyal" if r.tenure >= 40 else "New") + " " +
            ("high-spend" if r.MonthlyCharges >= 60 else "low-spend")
         for s, r in means.iterrows()}
print("Names:", names)
assert len(set(names.values())) == 4, "Names are not unique, tell me"

scored["segment_name"] = pd.Series(km4.labels_, index=X_seg.index).map(names)

summary = scored.groupby("segment_name").agg(
    customers=("Churn", "size"),
    churn_rate_pct=("Churn", lambda s: round((s == "Yes").mean() * 100, 1)),
    avg_churn_probability=("churn_probability", "mean"),
)
summary["avg_churn_probability"] = summary["avg_churn_probability"].round(3)
display(summary)

print("\nSegment vs risk level (customer counts):")
print(pd.crosstab(scored["segment_name"], scored["risk_level"]))

scored.to_csv(folder + r"\Customer_Churn_Scored_Segments.csv", index=False)
print("\nSaved: Customer_Churn_Scored_Segments.csv")

In [ ]:
seg = scored[["segment_name", "Contract", "InternetService", "PaymentMethod"]]

for col in ["Contract", "InternetService", "PaymentMethod"]:
    print(f"--- {col} (% of customers inside each segment) ---")
    display((pd.crosstab(seg["segment_name"], seg[col], normalize="index") * 100).round(1))

In [ ]:
rev = scored.copy()
rev["left"] = (rev["Churn"] == "Yes")
rev["lost_monthly"] = rev["MonthlyCharges"].where(rev["left"], 0)

def revenue_table(by):
    t = rev.groupby(by, observed=True).agg(
        customers=("left", "size"),
        monthly_revenue=("MonthlyCharges", "sum"),
        monthly_revenue_lost=("lost_monthly", "sum"),
    )
    t["pct_of_revenue_lost"] = t["monthly_revenue_lost"] / t["monthly_revenue"] * 100
    t["share_of_total_loss_pct"] = t["monthly_revenue_lost"] / t["monthly_revenue_lost"].sum() * 100
    return t.round(1)

print("Total customers:", len(rev))
print("Total monthly revenue:", round(rev["MonthlyCharges"].sum(), 0))
print("Monthly revenue lost to churn:", round(rev["lost_monthly"].sum(), 0))
print("Customers lost (%):", round(rev["left"].mean() * 100, 1))
print("Revenue lost (%):", round(rev["lost_monthly"].sum() / rev["MonthlyCharges"].sum() * 100, 1))

print("\n--- By segment ---")
display(revenue_table("segment_name"))
print("--- By risk level ---")
display(revenue_table("risk_level"))

In [ ]:
active = scored[scored["Churn"] == "No"].copy()
watch = active[active["risk_level"] == "High"].copy()
watch = watch.rename(columns={"churn_probability": "risk_score"})
watch = watch.sort_values("risk_score", ascending=False)

print("Active customers:", len(active))
print("Active High-risk customers:", len(watch))
print("Their monthly revenue:", round(watch["MonthlyCharges"].sum(), 0))
print("Share of active monthly revenue (%):",
      round(watch["MonthlyCharges"].sum() / active["MonthlyCharges"].sum() * 100, 1))

print("\nBy segment:")
display(watch.groupby("segment_name").agg(
    customers=("customerID", "size"),
    monthly_revenue=("MonthlyCharges", "sum"),
).round(0))

watch.to_csv(folder + r"\Retention_Watchlist.csv", index=False)
print("\nSaved: Retention_Watchlist.csv")

print("\nTop 10 by risk score:")
display(watch[["customerID", "segment_name", "Contract", "MonthlyCharges",
               "tenure", "risk_score"]].head(10))

In [ ]:
import pandas as pd

folder = r"C:\Users\acer\OneDrive\Desktop\ZIDIO\churn_customer_prediction"
scored = pd.read_csv(folder + r"\Customer_Churn_Scored_Segments.csv")
print(scored.shape)

In [ ]:
import pandas as pd

folder = r"C:\Users\acer\OneDrive\Desktop\ZIDIO\churn_customer_prediction"
scored = pd.read_csv(folder + r"\Customer_Churn_Scored_Segments.csv")
print(scored.shape)

active = scored[scored["Churn"] == "No"].copy()
watch = active[active["risk_level"] == "High"].copy()
watch = watch.rename(columns={"churn_probability": "risk_score"})
watch = watch.sort_values("risk_score", ascending=False)

print("\nActive customers:", len(active))
print("Active High-risk customers:", len(watch))
print("Their monthly revenue:", round(watch["MonthlyCharges"].sum(), 0))
print("Share of active monthly revenue (%):",
      round(watch["MonthlyCharges"].sum() / active["MonthlyCharges"].sum() * 100, 1))

print("\nBy segment:")
display(watch.groupby("segment_name").agg(
    customers=("customerID", "size"),
    monthly_revenue=("MonthlyCharges", "sum"),
).round(0))

watch.to_csv(folder + r"\Retention_Watchlist.csv", index=False)
print("\nSaved: Retention_Watchlist.csv")

print("\nTop 10 by risk score:")
display(watch[["customerID", "segment_name", "Contract", "MonthlyCharges",
               "tenure", "risk_score"]].head(10))


In [ ]:
import pandas as pd

folder = r"C:\Users\acer\OneDrive\Desktop\ZIDIO\churn_customer_prediction"
scored = pd.read_csv(folder + r"\Customer_Churn_Scored_Segments.csv")

m = scored[scored["Contract"] == "Month-to-month"].copy()
m["left"] = (m["Churn"] == "Yes").astype(int)
m["lost"] = m["MonthlyCharges"].where(m["left"] == 1, 0)

print("Customers:", len(m), "| Revenue lost:", round(m["lost"].sum(), 2))

for col in ["segment_name", "risk_level"]:
    t = m.groupby(col).agg(
        customers=("left", "size"),
        churn_pct=("left", lambda s: round(s.mean() * 100, 2)),
        revenue_lost=("lost", "sum"),
    ).round(2)
    print("\nBy", col)
    display(t)

In [ ]:
%pip install lightgbm

In [ ]:
import sys
print("Python:", sys.version.split()[0])

for name in ["tensorflow", "lightgbm"]:
    try:
        mod = __import__(name)
        print(name, "version:", mod.__version__)
    except Exception as e:
        print(name, "NOT available:", str(e)[:150])

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier

folder = r"C:\Users\acer\OneDrive\Desktop\ZIDIO\churn_customer_prediction"
df_feat = pd.read_csv(folder + r"\Customer_Churn_Features.csv")

X = df_feat.drop(columns=["Churn"])
y = df_feat["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

spw = (y_train == 0).sum() / (y_train == 1).sum()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"roc_auc": "roc_auc", "recall": "recall", "precision": "precision", "f1": "f1"}

models = {
    "Decision Tree (depth 4, balanced)": DecisionTreeClassifier(
        max_depth=4, class_weight="balanced", random_state=42),
    "LightGBM (weighted)": LGBMClassifier(
        n_estimators=200, learning_rate=0.05, num_leaves=8, max_depth=3,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        scale_pos_weight=spw, random_state=42, verbose=-1),
}

rows = []
for name, model in models.items():
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({
        "Model": name,
        "ROC-AUC": round(res["test_roc_auc"].mean(), 4),
        "Recall (churn)": round(res["test_recall"].mean(), 4),
        "Precision (churn)": round(res["test_precision"].mean(), 4),
        "F1 (churn)": round(res["test_f1"].mean(), 4),
    })

display(pd.DataFrame(rows))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score

folder = r"C:\Users\acer\OneDrive\Desktop\ZIDIO\churn_customer_prediction"
df_feat = pd.read_csv(folder + r"\Customer_Churn_Features.csv")

X = df_feat.drop(columns=["Churn"])
y = df_feat["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "services_count", "avg_monthly_spend"]
prep = ColumnTransformer([("scale", StandardScaler(), num_cols)], remainder="passthrough")

ann = make_pipeline(
    prep,
    MLPClassifier(hidden_layer_sizes=(32, 16), alpha=0.001, early_stopping=True,
                  validation_fraction=0.15, max_iter=500, random_state=42),
)

oof = cross_val_predict(ann, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
threshold = y_train.mean()
pred = (oof >= threshold).astype(int)

print("Threshold used:", round(threshold, 4))
print("ROC-AUC  :", round(roc_auc_score(y_train, oof), 4))
print("Recall   :", round(recall_score(y_train, pred), 4))
print("Precision:", round(precision_score(y_train, pred), 4))
print("F1       :", round(f1_score(y_train, pred), 4))

In [ ]:
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

folder = r"C:\Users\acer\OneDrive\Desktop\ZIDIO\churn_customer_prediction"
df = pd.read_csv(folder + r"\Customer_Churn_Features.csv")

seg_cols = ["tenure", "MonthlyCharges", "services_count"]
Xs = StandardScaler().fit_transform(df[seg_cols])

rows = []
for eps in [0.15, 0.2, 0.3, 0.4, 0.5]:
    labels = DBSCAN(eps=eps, min_samples=15).fit_predict(Xs)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise = int((labels == -1).sum())
    rows.append({"eps": eps, "clusters": n_clusters,
                 "noise_customers": noise,
                 "noise_pct": round(noise / len(labels) * 100, 1)})

display(pd.DataFrame(rows))

In [ ]:
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

folder = r"C:\Users\acer\OneDrive\Desktop\ZIDIO\churn_customer_prediction"
feat = pd.read_csv(folder + r"\Customer_Churn_Features.csv")
clean = pd.read_csv(folder + r"\Customer_Churn_Python_Clean.csv")

seg_cols = ["tenure", "MonthlyCharges", "services_count"]
Xs = StandardScaler().fit_transform(feat[seg_cols])
labels = DBSCAN(eps=0.3, min_samples=15).fit_predict(Xs)
feat["unusual"] = (labels == -1)

out = feat.groupby("unusual").agg(
    customers=("Churn", "size"),
    avg_tenure=("tenure", "mean"),
    avg_monthly_charge=("MonthlyCharges", "mean"),
    avg_services=("services_count", "mean"),
    churn_rate_pct=("Churn", lambda s: s.mean() * 100),
).round(1)
out.index = out.index.map({False: "Regular", True: "Unusual (noise)"})
display(out)

unusual = clean[feat["unusual"].values][
    ["customerID", "Contract", "InternetService", "PaymentMethod",
     "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
]
unusual.to_csv(folder + r"\Unusual_Customers.csv", index=False)
print("Saved Unusual_Customers.csv with", len(unusual), "customers")
display(unusual.head(8))

In [ ]:
pip install tensorflow

In [ ]:
import tensorflow as tf
import keras
import pandas as pd
import sklearn

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

In [ ]:
import numpy as np
import pandas as pd
import keras
from keras import layers
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score

keras.utils.set_random_seed(42)

folder = r"C:\Users\acer\OneDrive\Desktop\ZIDIO\churn_customer_prediction"
df_feat = pd.read_csv(folder + r"\Customer_Churn_Features.csv")

X = df_feat.drop(columns=["Churn"])
y = df_feat["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "services_count", "avg_monthly_spend"]
spw = float((y_train == 0).sum() / (y_train == 1).sum())
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def build(n_features):
    model = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="binary_crossentropy")
    return model

X_arr = X_train.reset_index(drop=True)
y_arr = y_train.reset_index(drop=True)
oof = np.zeros(len(X_arr))

for fold, (tr, va) in enumerate(cv.split(X_arr, y_arr), 1):
    Xtr, Xva = X_arr.iloc[tr].copy(), X_arr.iloc[va].copy()
    sc = StandardScaler().fit(Xtr[num_cols])
    Xtr[num_cols] = sc.transform(Xtr[num_cols])
    Xva[num_cols] = sc.transform(Xva[num_cols])

    model = build(Xtr.shape[1])
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                       restore_best_weights=True)
    h = model.fit(Xtr.values.astype("float32"), y_arr.iloc[tr].values,
                  epochs=100, batch_size=64, validation_split=0.15,
                  class_weight={0: 1.0, 1: spw}, callbacks=[es], verbose=0)
    oof[va] = model.predict(Xva.values.astype("float32"), verbose=0).ravel()
    print("fold", fold, "done | epochs run:", len(h.epoch))

pred = (oof >= 0.5).astype(int)
print("\nROC-AUC  :", round(roc_auc_score(y_arr, oof), 4))
print("Recall   :", round(recall_score(y_arr, pred), 4))
print("Precision:", round(precision_score(y_arr, pred), 4))
print("F1       :", round(f1_score(y_arr, pred), 4))